In [ ]:
from langchain_core.runnables import RunnableLambda

def add_one(x: int) -> int:
    return x + 1

runnable = RunnableLambda(add_one)
print("runnable : ",runnable)
print("runnable.get_input_jsonschema() : ",runnable.get_input_jsonschema())

In [ ]:
from langchain_core.runnables import RunnableLambda

def add_one(x: int) -> int:
    return x + 1

def mul_two(x: int) -> int:
    return x * 2

runnable_1 = RunnableLambda(add_one)
runnable_2 = RunnableLambda(mul_two)
sequence = runnable_1.pipe(runnable_2)
# Or equivalently:
# sequence = runnable_1 | runnable_2
# sequence = RunnableSequence(first=runnable_1, last=runnable_2)
sequence.invoke(1)
await sequence.ainvoke(1)
# -> 4

sequence.batch([1, 2, 3])
await sequence.abatch([1, 2, 3])
# -> [4, 6, 8]

In [ ]:
from langchain_community.chat_models import ChatOllama
from langchain_core.runnables import RunnableLambda

model="llama3.2:1b"

llm = ChatOllama(model=model)

uppercase = RunnableLambda(lambda x: x.upper())

chain = uppercase | llm

print(chain.invoke("hello agent"))


In [ ]:
# --------------------------------------------------------------
# 0️⃣ (If you haven't installed yet) --------------------------------------------------------------
# pip install --upgrade "langchain[all]" "langchain-ollama" "langchain-community"
# --------------------------------------------------------------

# --------------------------------------------------------------
# 1️⃣ Imports – note the new locations
# --------------------------------------------------------------
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnableBranch,
    RunnableBinding,
)
import json
import asyncio
import sys
from itertools import islice

# --------------------------------------------------------------
# 2️⃣ Helper that works both in scripts and notebooks
# --------------------------------------------------------------
def run_async(coro):
    """
    Run an async coroutine safely regardless of the surrounding environment.
    Returns the coroutine result in a script, or a Task in an already‑running loop.
    """
    # If we are inside a Jupyter/IPython cell we can `await` directly.
    if "IPython" in sys.modules:
        try:
            ip = get_ipython()
            # `run_cell_async` returns a coroutine that we can await.
            # In a regular notebook cell you can just write `await coro`.
            # Here we inject `await` into the cell so the user sees the result.
            ip.run_cell_async(f"await {coro!r}")
            return None  # the result is printed by the cell, no need to return
        except Exception:
            pass                        # fall back to generic handling

    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:                # no loop → normal script
        return asyncio.run(coro)
    else:                               # loop already running (e.g. some framework)
        # schedule and return a Task; caller can `await` it.
        return loop.create_task(coro)

# --------------------------------------------------------------
# 3️⃣ Create the Ollama LLM (Runnable)
# --------------------------------------------------------------
ollama_llm = ChatOllama(
    model="llama3.2:1b",          # change to any model you have locally
    temperature=0.0,
    max_tokens=1024,
)

# --------------------------------------------------------------
# 4️⃣ Core building blocks (prompt, parser, formatter)
# --------------------------------------------------------------
prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful assistant. Answer **exactly** in JSON with the keys:
    - "answer": a short plain‑text answer
    - "sources": a list of strings (may be empty)

    Question: {question}
    """
)

def json_parser(raw: str) -> dict:
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"answer": raw, "sources": []}

parser = RunnableLambda(json_parser)

def pretty(data: dict) -> str:
    src = ", ".join(data["sources"]) or "none"
    return f"Answer: {data['answer']}\nSources: {src}"

formatter = RunnableLambda(pretty)

# --------------------------------------------------------------
# 5️⃣ Wire the pieces together (| = sequence)
# --------------------------------------------------------------
core_pipeline = (
    prompt
    | ollama_llm                # returns AIMessage
    | StrOutputParser()         # → plain string
    | parser
    | formatter
)

# --------------------------------------------------------------
# 6️⃣ Demo – sync, async, streaming
# --------------------------------------------------------------
print("\n=== SYNC CALL ===")
print(core_pipeline.invoke({"question": "What is the capital of Italy?"}))

print("\n=== ASYNC CALL ===")
async_coro = core_pipeline.ainvoke(
    {"question": "Explain diffusion models in one sentence."}
)
# `run_async` works everywhere
async_result = run_async(async_coro)
# In a script `async_result` already holds the value.
# In a notebook the result is printed by the cell, so we ignore the return.
if async_result is not None:
    # we are in a pure script → get the value
    print(async_result)

print("\n=== STREAM (first 15 token chunks) ===")
stream = ollama_llm.stream(
    {"messages": [{"role": "user", "content": "Tell me a funny cat story"}]}
)
for chunk in islice(stream, 15):
    print(chunk.content, end="", flush=True)
print("\n--- end of token preview ---")

# --------------------------------------------------------------
# 7️⃣ Parallel tool calls + LLM synthesis
# --------------------------------------------------------------
search_tool = RunnableLambda(lambda p: f"Fake search result for '{p['question']}'")
vector_tool = RunnableLambda(lambda p: {"embedding": f"vec({p['question']})"})

parallel = search_tool + vector_tool   # parallel dict merge

combine_prompt = ChatPromptTemplate.from_template(
    """
    You have two pieces of information:

    1️⃣ Search result: {search}
    2️⃣ Vector embedding: {lookup[embedding]}

    Using ONLY these, answer the user query:
    {question}
    """
)

parallel_pipeline = (
    parallel
    | combine_prompt
    | ollama_llm
    | StrOutputParser()
    | formatter
)

print("\n=== PARALLEL + LLM ===")
print(parallel_pipeline.invoke(
    {"question": "What colors are on the flag of Japan?"}
))

# --------------------------------------------------------------
# 8️⃣ Conditional branching – weather vs generic answer
# --------------------------------------------------------------
def is_weather(payload: dict) -> bool:
    return "weather" in payload["question"].lower()

weather_tool = RunnableLambda(
    lambda p: f"The weather in {p['question'].split('in')[-1].strip()} is sunny."
)

fallback_prompt = ChatPromptTemplate.from_template(
    "Answer directly: {question}"
)
fallback = fallback_prompt | ollama_llm | StrOutputParser()

branch = RunnableBranch(
    predicate=is_weather,
    true_runnable=weather_tool,
    false_runnable=fallback,
)

top_prompt = ChatPromptTemplate.from_template(
    "User asked: {question}\n\nResponse:"
)

branch_pipeline = top_prompt | branch | formatter

print("\n=== BRANCH (weather) ===")
print(branch_pipeline.invoke({"question": "What is the weather in Paris?"}))
print("\n=== BRANCH (generic) ===")
print(branch_pipeline.invoke({"question": "Who invented the telephone?"}))

# --------------------------------------------------------------
# 9️⃣ OPTIONAL – tracing / naming (useful while developing)
# --------------------------------------------------------------
from langchain_core.tracing import LangChainTracer

tracer = LangChainTracer()   # prints a minimal trace to stdout
core_pipeline = core_pipeline.with_config(callbacks=[tracer])

print("\n=== TRACE DEMO ===")
core_pipeline.invoke({"question": "What is 7 * 8?"})
